# Ddri Holiday API Fetch

- 목적: 한국천문연구원 특일 정보 API에서 2023~2025 공휴일 데이터를 수집한다.
- 출력: 공휴일 원천 CSV, 일 단위 캘린더 CSV


## 1. Setup

In [1]:
from pathlib import Path
import re

import pandas as pd
import requests

BASE_DIR = Path('/Users/cheng80/Desktop/ddri_work')
API_DIR = BASE_DIR / '3조 공유폴더' / '[일정데이터] 특일 정보 API'
OUTPUT_DIR = BASE_DIR / 'works' / '02_data_collection' / '01_calendar' / 'data'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = 'https://apis.data.go.kr/B090041/openapi/service/SpcdeInfoService/getRestDeInfo'


## 2. API 키 읽기

In [2]:
def read_service_key():
    text = (API_DIR / 'API 인증키.txt').read_text().strip()
    match = re.search(r'일반 인증키\s*:\s*(\S+)', text)
    if not match:
        raise ValueError('API 인증키 파일에서 일반 인증키를 찾지 못했습니다.')
    return match.group(1)

service_key = read_service_key()
len(service_key)

64

## 3. 월별 공휴일 수집

In [3]:
def fetch_month(year, month, service_key):
    params = {
        'solYear': f'{year:04d}',
        'solMonth': f'{month:02d}',
        'ServiceKey': service_key,
        '_type': 'json',
        'numOfRows': '100',
    }
    response = requests.get(BASE_URL, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()
    body = data.get('response', {}).get('body', {})
    items = body.get('items', {})
    if isinstance(items, str):
        return []
    item = items.get('item', [])
    if isinstance(item, str):
        return []
    if isinstance(item, dict):
        item = [item]
    return item

years = [2023, 2024, 2025]
rows = []
for year in years:
    for month in range(1, 13):
        items = fetch_month(year, month, service_key)
        for item in items:
            rows.append({
                'date': pd.to_datetime(str(item['locdate']), format='%Y%m%d'),
                'locdate': str(item['locdate']),
                'date_name': item.get('dateName'),
                'date_kind': item.get('dateKind'),
                'is_holiday_api': item.get('isHoliday'),
                'seq': item.get('seq'),
                'source_year': year,
                'source_month': month,
            })

holiday_df = pd.DataFrame(rows).sort_values(['date', 'seq']).reset_index(drop=True)
holiday_df.head()

,date,locdate,date_name,date_kind,is_holiday_api,seq,source_year,source_month
0,2023-01-01,20230101,1월1일,01,Y,1,2023,1
1,2023-01-21,20230121,설날,01,Y,1,2023,1
2,2023-01-22,20230122,설날,01,Y,1,2023,1
3,2023-01-23,20230123,설날,01,Y,1,2023,1
4,2023-01-24,20230124,대체공휴일,01,Y,1,2023,1


## 4. 일 단위 캘린더 테이블 생성

In [4]:
all_dates = pd.date_range('2023-01-01', '2025-12-31', freq='D')
calendar_df = pd.DataFrame({'date': all_dates})
calendar_df['year'] = calendar_df['date'].dt.year
calendar_df['month'] = calendar_df['date'].dt.month
calendar_df['day'] = calendar_df['date'].dt.day
calendar_df['day_of_week'] = calendar_df['date'].dt.dayofweek
calendar_df['is_weekend'] = (calendar_df['day_of_week'] >= 5).astype(int)

holiday_daily = (
    holiday_df.groupby('date')
    .agg(
        holiday_name=('date_name', lambda x: ' | '.join(sorted(set(str(v) for v in x)))),
        holiday_count=('date_name', 'size'),
        is_holiday=('is_holiday_api', lambda x: int(any(str(v) == 'Y' for v in x))),
    )
    .reset_index()
)
calendar_df = calendar_df.merge(holiday_daily, on='date', how='left')
calendar_df['holiday_count'] = calendar_df['holiday_count'].fillna(0).astype(int)
calendar_df['is_holiday'] = calendar_df['is_holiday'].fillna(0).astype(int)
calendar_df['holiday_name'] = calendar_df['holiday_name'].fillna('')
calendar_df['is_business_holiday'] = ((calendar_df['is_weekend'] == 1) | (calendar_df['is_holiday'] == 1)).astype(int)
calendar_df.head()

,date,year,month,day,day_of_week,is_weekend,holiday_name,holiday_count,is_holiday,is_business_holiday
0,2023-01-01,2023,1,1,6,1,1월1일,1,1,1
1,2023-01-02,2023,1,2,0,0,,0,0,0
2,2023-01-03,2023,1,3,1,0,,0,0,0
3,2023-01-04,2023,1,4,2,0,,0,0,0
4,2023-01-05,2023,1,5,3,0,,0,0,0


## 5. 저장

In [5]:
holiday_path = OUTPUT_DIR / 'ddri_holiday_api_raw_2023_2025.csv'
calendar_path = OUTPUT_DIR / 'ddri_calendar_daily_2023_2025.csv'
holiday_df.to_csv(holiday_path, index=False)
calendar_df.to_csv(calendar_path, index=False)
holiday_path, calendar_path

(PosixPath('/Users/cheng80/Desktop/ddri_work/works/02_data_collection/01_calendar/data/ddri_holiday_api_raw_2023_2025.csv'),
 PosixPath('/Users/cheng80/Desktop/ddri_work/works/02_data_collection/01_calendar/data/ddri_calendar_daily_2023_2025.csv'))